1. Проверим таблицы в PostgreSQL

In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="oil_db",
    user="analyst",
    password="secret"
)

tables = ['wells', 'production', 'well_telemetry', 'well_targets', 
          'pumps', 'pump_sensors', 'pump_failures', 'deliveries', 'drivers', 'vehicles', 'oil_stations']

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    print(f"{table}: {df['cnt'].iloc[0]} записей")

wells: 5 записей
production: 150 записей
well_telemetry: 48 записей
well_targets: 90 записей
pumps: 5 записей
pump_sensors: 72 записей
pump_failures: 3 записей
deliveries: 30 записей
drivers: 5 записей
vehicles: 5 записей
oil_stations: 20 записей


/tmp/ipykernel_126/319351375.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)


2. Подключение к MinIO, создание бакета

In [2]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3.create_bucket(Bucket='oil-data')
    print("Бакет 'oil-data' создан")
except Exception as e:
    print(f"Error: {e}")

buckets = s3.list_buckets()
print("Список бакетов:", [b['Name'] for b in buckets['Buckets']])

Бакет 'oil-data' создан
Список бакетов: ['oil-data']


3. Выгрузка данных в MinIO (Parquet + партиционирование)

In [3]:
from io import BytesIO

# Загрузка production с данными о скважинах
df_prod = pd.read_sql("""
    SELECT p.*, w.name as well_name, w.region, w.field_name, w.operator
    FROM production p
    JOIN wells w ON p.well_id = w.well_id
""", conn)

print(f"Загружено {len(df_prod)} записей")

# Преобразование даты 
df_prod['date'] = pd.to_datetime(df_prod['date'])

# Партиционирование по дате (год/месяц/день). Довольно мелкое разбиение, но наглядное.
for (year, month, day), group in df_prod.groupby([df_prod['date'].dt.year, df_prod['date'].dt.month, df_prod['date'].dt.day]):
    key = f"oil-data/production/year={year}/month={month}/day={day}/data.parquet"
    
    buffer = BytesIO()
    group.to_parquet(buffer, index=False)
    buffer.seek(0)
    
    s3.put_object(Bucket='oil-data', Key=key, Body=buffer.getvalue())
    print(f" Сохранено: {key} ({len(group)} записей)")


/tmp/ipykernel_126/1811651525.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_prod = pd.read_sql("""


Загружено 150 записей
 Сохранено: oil-data/production/year=2025/month=10/day=1/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=2/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=3/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=4/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=5/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=6/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=7/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=8/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=9/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=10/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=11/data.parquet (5 записей)
 Сохранено: oil-data/production/year=2025/month=10/day=12/data.parquet (5 

4. Обработка NULL, фильтрация выбросов и создание фичей.
   В предоставленных к заданию данных NULL значения для temperature и pressure имеются лишь у одной скважины, для корторой добыча не велась на протяжении всего месяца. Однако, это не значит, что там не существовало давления и температуры, поэтому заменим их на медианные значения по другим скважинам, заодно продемонстрируем обработку NULL.

In [4]:
# Обработка NULL
df_prod['temperature'] = df_prod['temperature'].fillna(df_prod['temperature'].median())
df_prod['pressure'] = df_prod['pressure'].fillna(df_prod['pressure'].median())

# Фильтрация выбросов 
df_prod = df_prod[df_prod['oil_ton'] >= 0]
print(f"После фильтрации: {len(df_prod)} записей")

# Создание фичей

# Среднее давление по скважине за день
df_prod['avg_pressure'] = df_prod.groupby(['well_id', df_prod['date'].dt.date])['pressure'].transform('mean')

# Средняя температура по скважине за день
df_prod['avg_temperature'] = df_prod.groupby(['well_id', df_prod['date'].dt.date])['temperature'].transform('mean')

# Коэффициент простоя
df_prod['downtime_ratio'] = df_prod['downtime_hours'] / 24

# Результат
print("\n Результат:")
print(df_prod[['well_id', 'well_name', 'date', 'oil_ton', 'avg_pressure', 'avg_temperature', 'downtime_ratio']].head(10))

После фильтрации: 150 записей

 Результат:
   well_id well_name       date  oil_ton  avg_pressure  avg_temperature  \
0        1  Well-101 2025-10-01    212.4         120.4             88.1   
1        1  Well-101 2025-10-02    213.8         121.0             87.8   
2        1  Well-101 2025-10-03    211.9         119.8             88.5   
3        1  Well-101 2025-10-04    215.1         121.5             87.6   
4        1  Well-101 2025-10-05    214.6         120.9             88.0   
5        1  Well-101 2025-10-06    213.2         120.3             88.2   
6        1  Well-101 2025-10-07    211.7         119.6             89.1   
7        1  Well-101 2025-10-08    212.5         120.1             88.4   
8        1  Well-101 2025-10-09    213.9         121.2             88.0   
9        1  Well-101 2025-10-10    212.0         120.0             88.7   

   downtime_ratio  
0        0.020833  
1        0.012500  
2        0.029167  
3        0.008333  
4        0.016667  
5        0.